# Compile State Stats

Stage 2 of the data pipeline. Reads the CSVs produced by `pipeline.ipynb` from `output_csvs/`, 
filters each to the latest year and residential sector where applicable, 
and joins everything into a single state-level summary table (`output_csvs/state_stats.csv`).

In [ ]:
# Median solar cost and system size (LBNL Tracking the Sun, 2024)
solar_costs = pd.read_csv("../output_csvs/residential_solar_costs_by_state_over_time.csv")
solar_costs = solar_costs[solar_costs["year"] == 2024][["state", "median_price_per_kw", "median_system_size_kw"]]


In [2]:
# Bill savings (2026 cohort, baseline scenario)
savings = load_from_export("../data/bill_savings_csvs")
savings = savings.rename(columns={"state_abbr": "state"})
savings = savings[["state", "year_1_savings", "lifetime_savings"]]


In [ ]:
# Residential electricity prices (from output_csvs, latest year, residential sector)
elec = pd.read_csv("../output_csvs/electricity_rates_by_state.csv")
elec = elec[elec["sector"] == "Residential"]
elec = elec[elec["year"] == elec["year"].max()]
elec = elec[["state", "price_per_kwh", "avg_annual_bill"]].rename(columns={"avg_annual_bill": "annual_bill"})

In [ ]:
# Solar capacity/installations (from output_csvs, latest year, residential sector)
solar = pd.read_csv("../output_csvs/solar_storage_capacity_installations_by_state_sector.csv")
solar = solar[solar["sector"] == "Residential"]
solar = solar[solar["year"] == solar["year"].max()]
solar = solar[["state", "pv_customers", "pv_capacity_mw"]]

In [5]:
# Solar-eligible households (ResStock filter only)
# Rooftop suitability haircut is NOT applied in this run.
# To include it, uncomment the apply_rooftop_suitability line below.

resstock = pd.read_csv("../data/resstock_metadata_technical_potential.csv")
eligibility = compute_solar_eligibility(resstock)
# eligibility = apply_rooftop_suitability(eligibility)

In [6]:
# Residential PV technical generation potential by state (NREL SLOPE, 2020)
techpot = pd.read_csv("../output_csvs/potential_solar_generation_by_state.csv")


In [7]:
# Median solar cost and system size (LBNL Tracking the Sun, 2024)
solar_costs = load_median_solar_costs_and_size(fred_api_key, year=2024)

/Users/wael/Documents/repos/basic-solar-statistics/notebooks/../src/median_solar_costs.py:288: DtypeWarning: Columns (1,2,3,11,15,16,18,28,29,31,32,34,35,38,39,40,53,54,56,57,59,60,74,75,79,80) have mixed types. Specify dtype option on import or set low_memory=False.
  lbnl = pd.read_csv(lbnl_path)


In [8]:
# Cancellation rates (2022-2024 combined, min 500 applications)
cancel = pd.read_csv("../data/state_cancellation_rates.csv")
cancel = cancel[(cancel["year"] == "2022_2024") & (cancel["num_applications"] >= 500)]
cancel = cancel.merge(name_abbr, left_on="state", right_on="state", how="inner")
cancel = cancel[["state_abbr", "pct_cancelled"]].rename(columns={"state_abbr": "state"})

In [9]:
# Permitting timelines by technology (2024, min 500 records)
permit = pd.read_csv("../data/state_median_permitting_timelines_by_tech.csv")
permit = permit[(permit["year"] == 2024) & (permit["num_records"] >= 500)]
permit = permit.merge(name_abbr, left_on="state", right_on="state", how="inner")

solar_permit = (permit[permit["technology"] == "solar"]
    [["state_abbr", "median_permitting_timeline_days"]]
    .rename(columns={"state_abbr": "state", "median_permitting_timeline_days": "median_permit_days_solar"}))

storage_permit = (permit[permit["technology"] == "solar_storage"]
    [["state_abbr", "median_permitting_timeline_days"]]
    .rename(columns={"state_abbr": "state", "median_permitting_timeline_days": "median_permit_days_solar_storage"}))

permit_df = solar_permit.merge(storage_permit, on="state", how="outer")

In [10]:
# Interconnection timelines (2023, min 500 installs, Final IX to PTO)
ix_raw = pd.read_csv("../data/solartrace_ix.csv")
_, state_ix = ix_run_pipeline(ix_raw)
state_ix = state_ix[(state_ix["year"] == 2023) & (state_ix["total_installs"] > 500)]

ix_pv = (state_ix[state_ix["tech"] == "pv_only"]
    [["state", "weighted_median_ix"]]
    .rename(columns={"weighted_median_ix": "median_ix_days_pv_only"}))

ix_storage = (state_ix[state_ix["tech"] == "pv_storage"]
    [["state", "weighted_median_ix"]]
    .rename(columns={"weighted_median_ix": "median_ix_days_pv_storage"}))

ix_df = ix_pv.merge(ix_storage, on="state", how="outer")

In [14]:
# Join all state-level stats
state_stats = (
    savings
    .merge(elec, on="state", how="outer")
    .merge(solar, on="state", how="outer")
    .merge(eligibility, on="state", how="outer")
    .merge(techpot, on="state", how="outer")
    .merge(solar_costs, on="state", how="outer")
    .merge(cancel, on="state", how="outer")
    .merge(permit_df, on="state", how="outer")
    .merge(ix_df, on="state", how="outer")
)

# Drop rows not in state mapping (e.g. territories, US totals)
state_stats = state_stats.merge(name_abbr.rename(columns={"state": "state_name"}), left_on="state", right_on="state_abbr", how="inner")
state_stats = state_stats.drop(columns=["state_abbr"]).rename(columns={"state": "state_abbr"})

# Solar penetration: solar installations / eligible households
state_stats["solar_penetration_pct"] = state_stats["pv_customers"] / state_stats["eligible_households"]

# Rename columns for final output
state_stats = state_stats.rename(columns={
    "state_name":                       "State",
    "state_abbr":                       "State (Abbr.)",
    "year_1_savings":                   "Median solar savings in first year",
    "lifetime_savings":                 "Median solar savings over lifetime",
    "price_per_kwh":                    "Average electricity retail cost ($/kWh)",
    "annual_bill":                      "Average annual electricity bill",
    "pv_customers":                     "Number of solar installations",
    "pv_capacity_mw":                   "Total installed solar capacity (MW)",
    "total_households":                 "Total households",
    "eligible_households":              "Solar eligible and suitable households",
    "eligible_pct":                     "Percent of total households suitable",
    "solar_penetration_pct":            "Solar penetration",
    "technical_potential_gwh":          "Annual technical potential (GWh)",
    "median_price_per_kw":              "Median solar cost ($/kW)",
    "median_system_size_kw":            "Median solar system size (kW)",
    "pct_cancelled":                    "Solar permit cancellation rate (2022-2024)",
    "median_permit_days_solar":         "Median permitting timeline - solar (days)",
    "median_permit_days_solar_storage": "Median permitting timeline - solar+storage (days)",
    "median_ix_days_pv_only":           "Median interconnection timeline - PV only (days)",
    "median_ix_days_pv_storage":        "Median interconnection timeline - PV+storage (days)",
})

# Final column order
state_stats = state_stats[[
    "State", "State (Abbr.)",
    "Median solar savings in first year", "Median solar savings over lifetime",
    "Average electricity retail cost ($/kWh)", "Average annual electricity bill",
    "Number of solar installations", "Total installed solar capacity (MW)",
    "Total households", "Solar eligible and suitable households",
    "Percent of total households suitable", "Solar penetration",
    "Annual technical potential (GWh)",
    "Median solar cost ($/kW)", "Median solar system size (kW)",
    "Solar permit cancellation rate (2022-2024)",
    "Median permitting timeline - solar (days)",
    "Median permitting timeline - solar+storage (days)",
    "Median interconnection timeline - PV only (days)",
    "Median interconnection timeline - PV+storage (days)",
]]

state_stats

,State,State (Abbr.),Median solar savings in first year,Median solar savings over lifetime,Average electricity retail cost ($/kWh),Average annual electricity bill,Number of solar installations,Total installed solar capacity (MW),Total households,Solar eligible and suitable households,Percent of total households suitable,Solar penetration,Annual technical potential (GWh),Median solar cost ($/kW),Median solar system size (kW),Solar permit cancellation rate (2022-2024),Median permitting timeline - solar (days),Median permitting timeline - solar+storage (days),Median interconnection timeline - PV only (days),Median interconnection timeline - PV+storage (days)
0,Alaska,AK,NaN,NaN,0.242632,1680.980948,3004.0,16.227,3.186491e+05,1.508188e+05,0.473307,0.019918,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Alabama,AL,953.758080,42926.396254,0.151934,2091.362533,0.0,4.458,2.301637e+06,1.128348e+06,0.490237,0.000000,10741.47579,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Arkansas,AR,824.670133,36657.734014,0.123168,1547.934086,16021.0,180.206,1.398248e+06,6.769072e+05,0.484111,0.023668,6529.45901,NaN,NaN,0.1568,NaN,NaN,NaN,NaN
3,Arizona,AZ,2038.509747,91220.747140,0.149292,1867.727705,328233.0,2443.592,3.034657e+06,1.452583e+06,0.478665,0.225965,13993.68649,4046.826415,8.8000,0.1915,16.0,39.0,6.0,11.0
4,California,CA,1774.846810,81690.339880,0.244827,1358.503308,2193555.0,13795.564,1.449155e+07,6.632218e+06,0.457661,0.330742,70314.63613,4651.441885,5.8250,0.2313,5.0,3.0,12.0,8.0
5,Colorado,CO,1040.575022,48474.006062,0.148822,1188.792763,171846.0,1007.072,2.380093e+06,1.273073e+06,0.534884,0.134985,8612.71255,4401.575844,6.0000,0.1701,3.0,7.0,6.0,5.0
6,Connecticut,CT,1704.960285,77241.036971,0.234554,1951.901139,119433.0,934.825,1.556937e+06,8.759677e+05,0.562622,0.136344,5169.22788,5538.955219,7.0000,0.2196,4.0,NaN,3.0,NaN
7,District of Columbia,DC,NaN,NaN,0.149590,1138.007460,21479.0,141.148,3.196647e+05,8.937409e+04,0.279587,0.240327,156.98218,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Delaware,DE,1314.894999,58950.076292,0.157057,1696.615813,14054.0,114.279,4.354448e+05,2.338453e+05,0.537026,0.060100,1687.07653,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Florida,FL,1819.179594,81286.303092,0.141425,1868.942182,263275.0,2892.627,9.534083e+06,4.161227e+06,0.436458,0.063269,45803.02075,4558.918998,9.6150,0.2048,7.0,9.0,12.5,13.0


In [15]:
# Export to CSV (for copy-paste into Google Sheets)
state_stats.to_csv("../output_csvs/state_stats.csv", index=False)
print("Saved to output_csvs/state_stats.csv")

# # Upload to Google Drive
# folder_id = ensure_path(service, ROOT_ID, ["1. Solar"])
# upload_df_to_drive(state_stats, folder_id, "State Solar and Electricity Stats")

Saved to output_csvs/state_stats.csv
